# Title : Vortex Panel Method in 2D

**Submitted By**  

**Aathi Skantha Roopan T - SC26M001**

# Required packages

In [1]:
# do not change line below

# import required packages here
import numpy as np
import matplotlib.pyplot as plt
from numpy import sin, cos, pi, sqrt, arctan2
from numpy.linalg import solve

# Introduction

The vortex panel method is a numerical technique used to calculate the aerodynamic characteristics of two-dimensional airfoils and other bodies immersed in an inviscid, incompressible flow.

In this method, the surface of the body is divided into a finite number of straight-line panels. Vortex elements are distributed along these panels to represent the flow around the body.

The strength of the vortices determines the velocity field around the airfoil. By applying the no-penetration boundary condition at the control points and the Kutta condition at the trailing edge, a system of algebraic equations can be obtained and solved to determine the vortex strengths.

The pressure coefficient is then calculated using Bernoulli's equation.

Two formulations are considered in this study:

1. Constant-strength vortex panel method.
2. Linear-strength vortex panel method.

The methods are verified using a circular cylinder and tested on symmetric and cambered airfoils. Their convergence and limitations are also investigated.

**Assumptions**

The following assumptions are made in the vortex panel method:

1. The flow is two-dimensional.
2. The fluid is incompressible.
3. The flow is inviscid.
4. The flow is irrotational everywhere except at the vortex singularities.
5. The airfoil surface is impermeable.
6. The Kutta condition is applied at the trailing edge to obtain a physically meaningful circulation.

**Discretization of the body**

- The surface of the body is divided into N straight-line panels. 
- Each panel is defined by two consecutive nodal points.
- For panel j, the starting point is $(x_j, y_j)$ and the ending point is $(x_{j+1}, y_{j+1})$.
- The panel length is:
$$S_j=\sqrt{(x_{j+1}-x_j)^2+(y_{j+1}-y_j)^2}$$

- The panel inclination angle is:
$$\theta_j=\tan^{-1}\left(\frac{y_{j+1}-y_j}{x_{j+1}-x_j}\right)$$

- The control point is located at the midpoint of each panel:
$$x_{c,j}=\frac{x_j+x_{j+1}}{2}$$
$$y_{c,j}=\frac{y_j+y_{j+1}}{2}$$

The local coordinate system of each panel is used to calculate the induced velocity due to the vortex distribution.

The below code defines a function to calculate panel geometry : 

In [2]:
def panel_geometry(x, y):

    N = len(x) - 1

    dx = np.diff(x)
    dy = np.diff(y)

    S = np.sqrt(dx**2 + dy**2)

    theta = np.arctan2(dy, dx)

    xc = (x[:-1] + x[1:]) / 2
    yc = (y[:-1] + y[1:]) / 2

    return N, S, theta, xc, yc

**Boundary conditions**

The no-penetration boundary condition states that the normal component of velocity at the surface of the body must be zero.

Mathematically,

$$
\vec V \cdot \vec n=0
$$

where $\vec V$ is the total velocity and $\vec n$ is the outward unit normal vector.

The total velocity consists of the freestream velocity and the velocity induced by all vortex panels.

The Kutta condition requires the flow to leave the trailing edge smoothly.  

For a sharp trailing edge, this is commonly implemented by requiring the vortex strengths at the upper and lower trailing-edge endpoints to be equal in the linear formulation.

# Constant-strength vortex panel method

In the constant-strength vortex panel method, the vortex strength is assumed to remain constant over the entire length of each panel.

For panel j:
$$\gamma_j=\text{constant}$$


The unknown vortex strengths are determined by applying the no-penetration boundary condition at the control points.

The induced velocity at each control point is obtained by summing the contributions from all vortex panels.

The resulting equations can be written in matrix form as:
$$
[A]\{\gamma\}=\{b\}
$$

where:

- [A] is the influence coefficient matrix.
- {γ} is the vector of unknown vortex strengths.
- {b} represents the freestream velocity contribution.

After solving the linear system, the tangential velocity is calculated and the pressure coefficient is obtained from:

$$C_p=1-\left(\frac{V_t}{V_\infty}\right)^2$$


**Algorithm**  

1. Input the coordinates of the body surface and the number of panels.

2. Divide the body into straight-line panels.

3. Calculate the length, inclination angle and control point of each panel.

4. Calculate the unit normal and tangential directions.

5. Calculate the velocity influence coefficients of each vortex panel at every control point.

6. Construct the coefficient matrix using the no-penetration boundary condition.

7. Apply the Kutta condition at the trailing edge.

8. Solve the resulting system of linear algebraic equations to obtain the constant vortex strengths.

9. Calculate the tangential velocity at each control point.

10. Calculate the pressure coefficient using Bernoulli's equation.

11. Calculate the lift coefficient and plot the pressure coefficient distribution.

12. Return the vortex strengths, velocity and pressure coefficient.

**Mathematical formulation of influence coefficients**

**Velocity induced by a constant-strength vortex panel**

Consider a straight vortex panel of length $S_j$, having constant vortex strength $\gamma_j$.

The velocity induced at a point $x_i,y_i$ by a small vortex element is given by the Biot–Savart law:
$$
du=-\frac{\gamma_j}{2\pi}\frac{y_i-y}{(x_i-x)^2+(y_i-y)^2}ds
$$
$$
dv=\frac{\gamma_j}{2\pi}
\frac{x_i-x}{(x_i-x)^2+(y_i-y)^2}ds
$$

The total induced velocity is obtained by integrating along the panel:
$$
u_{ij}=-\frac{\gamma_j}{2\pi}\int_0^{S_j}\frac{y_i-y(s)}{r^2}ds
$$
$$
v_{ij}=\frac{\gamma_j}{2\pi}\int_0^{S_j}\frac{x_i-x(s)}{r^2}ds
$$

where
$$
r^2=(x_i-x(s))^2+(y_i-y(s))^2
$$

The total velocity at a control point is the sum of the freestream velocity and the velocities induced by all vortex panels.

The no-penetration condition is:
$$
V_{n,i}=0
$$

Therefore,
$$
\sum_{j=1}^{N} A_{ij}\gamma_j=-V_\infty\sin(\theta_i-\alpha)
$$

where $A_{ij}$ represents the normal velocity influence coefficient.

**Application of the Kutta condition**

or an airfoil with a sharp trailing edge, the Kutta condition requires the vortex strengths at the upper and lower trailing-edge panels to be equal and opposite:
$$
\gamma_1+\gamma_N=0
$$

For \(N\) constant-strength vortex panels, we have \(N\) unknowns. We use \(N-1\) no-penetration equations and one Kutta equation to obtain a square system:
$$
[A]\{\gamma\}=\{b\}
$$

The final row of the matrix represents the Kutta condition.

For a circular cylinder, there is no trailing edge. Instead, we impose zero net circulation:
$$
\Gamma=\sum_{j=1}^{N}\gamma_jS_j=0
$$

This ensures that the cylinder solution has no artificial circulation.

**Python implementation :**

In [3]:
def vortex_panel_method_0(x, y, V_inf=1.0, alpha=0.0,kutta=True):

    N = len(x) - 1

    # Panel geometry
    dx = np.diff(x)
    dy = np.diff(y)

    S = np.sqrt(dx**2 + dy**2)

    theta = np.arctan2(dy, dx)

    xc = (x[:-1] + x[1:]) / 2
    yc = (y[:-1] + y[1:]) / 2

    # Determine body orientation
    area = 0.5 * np.sum(
        x[:-1] * y[1:] - x[1:] * y[:-1]
    )

    orientation = 1 if area > 0 else -1

    # Outward unit normals
    nx = orientation * np.sin(theta)
    ny = -orientation * np.cos(theta)

    # Unit tangential vectors
    tx = np.cos(theta)
    ty = np.sin(theta)

    alpha = np.radians(alpha)

    U = V_inf * np.cos(alpha)
    V = V_inf * np.sin(alpha)

    # Numerical integration points
    nodes, weights = np.polynomial.legendre.leggauss(24)

    nodes = (nodes + 1) / 2
    weights = weights / 2

    # Influence coefficient matrix
    A = np.zeros((N, N))

    for i in range(N):

        for j in range(N):

            # Self-panel normal influence
            if i == j:
                A[i, j] = 0.0
                continue

            # Source points along panel j
            xs = x[j] + nodes * dx[j]
            ys = y[j] + nodes * dy[j]

            rx = xc[i] - xs
            ry = yc[i] - ys

            r2 = rx**2 + ry**2

            # Induced velocity per unit vortex strength
            u = np.sum(weights * (-ry / r2)) / (2 * np.pi) * S[j]

            v = np.sum(weights * (rx / r2)) / (2 * np.pi) * S[j]

            A[i, j] = u * nx[i] + v * ny[i]

    # Freestream normal velocity
    rhs = -(U * nx + V * ny)

    # Apply trailing-edge Kutta condition
    if kutta:

        A[-1, :] = 0.0
        A[-1, 0] = 1.0
        A[-1, -1] = 1.0

        rhs[-1] = 0.0

    # For a cylinder, impose zero net circulation
    else:

        A[-1, :] = S
        rhs[-1] = 0.0

    # Solve for vortex strengths
    gamma = np.linalg.solve(A, rhs)

    # Calculate tangential velocities
    Vt = U * tx + V * ty

    for i in range(N):

        for j in range(N):

            if i == j:

                Vt[i] += orientation * gamma[j] / 2

            else:

                xs = x[j] + nodes * dx[j]
                ys = y[j] + nodes * dy[j]

                rx = xc[i] - xs
                ry = yc[i] - ys

                r2 = rx**2 + ry**2

                u = np.sum(weights * (-ry / r2)) / (2 * np.pi) * S[j]

                v = np.sum(weights * (rx / r2)) / (2 * np.pi) * S[j]

                Vt[i] += gamma[j] * (u * tx[i] + v * ty[i])

    # Pressure coefficient
    Cp = 1 - (Vt / V_inf)**2

    # Total circulation
    circulation = np.sum(gamma * S)

    # Lift coefficient
    chord = np.max(x) - np.min(x)

    Cl = -2 * circulation / (V_inf * chord)

    return {
        "gamma": gamma,
        "Vt": Vt,
        "Cp": Cp,
        "Cl": Cl,
        "circulation": circulation,
        "xc": xc,
        "yc": yc,
        "S": S,
        "theta": theta
    }

# Linear-strength vortex panel method

In the linear-strength vortex panel method, the vortex strength is assumed to vary linearly between the two endpoints of each panel.

For panel $j$, the vortex strength is expressed as:
$$
\gamma_j(s)=\gamma_j+
\frac{s}{S_j}(\gamma_{j+1}-\gamma_j)
$$

where:
- $\gamma_j$ is the vortex strength at the starting endpoint.
- $\gamma_{j+1}$ is the vortex strength at the ending endpoint.
- $S_j$ is the panel length.
- $s$ is the distance measured along the panel.
The vortex strength is continuous between adjacent panels because they share a common endpoint value.

For $N$ panels, there are $N+1$ unknown nodal vortex strengths.

The unknowns are determined by applying the no-penetration condition at the $N$ control points and the Kutta condition at the trailing edge.

The final system contains $N+1$ equations.

The Kutta condition is:
$$
\boxed{\gamma_1+\gamma_{N+1}=0}
$$

The resulting vortex strengths are used to calculate tangential velocity and pressure coefficient.

**Algorithm**

1. Input the coordinates of the body surface and the number of panels.
2. Divide the body surface into straight-line panels.
3. Calculate the length, inclination angle and midpoint of each panel.
4. Define the vortex strength at each panel endpoint.
5. Assume that vortex strength varies linearly between adjacent endpoints.
6. Calculate the normal and tangential velocity influence coefficients using the Biot–Savart law.
7. Assemble the influence coefficient matrix using the no-penetration boundary condition at all control points.
8. Apply the Kutta condition at the trailing edge to obtain the additional equation.
9. Solve the resulting system of N+1 simultaneous equations to determine the nodal vortex strengths.
10. Calculate the tangential velocity at each control point.
11. Calculate the pressure coefficient using Bernoulli's equation.
12. Integrate the pressure distribution to obtain the lift coefficient.
13. Return the nodal vortex strengths, velocity, pressure coefficient and lift coefficient.

**Mathematical formulation of influence coefficients**

**Influence coefficients for linear-strength panels**

The velocity induced by a vortex distribution is obtained by integrating the Biot–Savart law along each panel.

The linear vortex distribution can be expressed using two linear interpolation functions:
$$
N_1(s)=1-\frac{s}{S_j}
$$

$$
N_2(s)=\frac{s}{S_j}
$$

Therefore,
$$
\gamma_j(s)=N_1(s)\gamma_j+N_2(s)\gamma_{j+1}
$$

The induced velocity is calculated by integrating the contributions associated with both endpoint strengths.

At each control point, the normal velocity must satisfy:
$$
V_{n,i}=0
$$

Combining the contributions from all panels gives:
$$
\sum_{j=1}^{N+1} A_{ij}\gamma_j=-V_\infty\vec e_\infty\cdot\vec n_i
$$

The Kutta condition provides the final equation:
$$
\gamma_1+\gamma_{N+1}=0
$$

Thus, the system can be represented as:
$$
[A]\{\gamma\}=\{b\}
$$

The system is solved to obtain all nodal vortex strengths.

**Python implementation :**

In [5]:
def vortex_panel_method_1(x, y, V_inf=1.0, alpha=0.0,
                          kutta=True):

    """
    Linear-strength vortex panel method.

    Parameters
    ----------
    x, y : arrays
        Coordinates of body surface nodes.
    V_inf : float
        Freestream velocity.
    alpha : float
        Angle of attack in degrees.
    kutta : bool
        True for airfoil, False for circular cylinder.

    Returns
    -------
    Dictionary containing nodal vortex strengths,
    tangential velocities, Cp and lift coefficient.
    """

    # Number of panels
    N = len(x) - 1

    # Panel geometry
    dx = np.diff(x)
    dy = np.diff(y)

    S = np.sqrt(dx**2 + dy**2)

    theta = np.arctan2(dy, dx)

    xc = (x[:-1] + x[1:]) / 2
    yc = (y[:-1] + y[1:]) / 2

    # Determine orientation
    area = 0.5 * np.sum(x[:-1] * y[1:] - x[1:] * y[:-1])

    orientation = 1 if area > 0 else -1

    # Outward normal vectors
    nx = orientation * np.sin(theta)
    ny = -orientation * np.cos(theta)

    # Tangential vectors
    tx = np.cos(theta)
    ty = np.sin(theta)

    # Freestream velocity components
    alpha = np.radians(alpha)

    U = V_inf * np.cos(alpha)
    V = V_inf * np.sin(alpha)

    # Gaussian quadrature
    nodes, weights = np.polynomial.legendre.leggauss(32)

    nodes = (nodes + 1) / 2
    weights = weights / 2

    # Influence matrices
    An = np.zeros((N, N+1))
    At = np.zeros((N, N+1))

    for i in range(N):

        for j in range(N):

            # Self-panel influence
            if i == j:

                At[i, j] += orientation / 4
                At[i, j+1] += orientation / 4

                continue

            # Integration points on panel j
            xs = x[j] + nodes * dx[j]
            ys = y[j] + nodes * dy[j]

            rx = xc[i] - xs
            ry = yc[i] - ys

            r2 = rx**2 + ry**2

            # Linear interpolation functions
            N1 = 1 - nodes
            N2 = nodes

            # Velocity influence of first endpoint
            u1 = np.sum(weights * N1 * (-ry / r2)) * S[j] / (2*pi)

            v1 = np.sum(weights * N1 * (rx / r2)) * S[j] / (2*pi)

            # Velocity influence of second endpoint
            u2 = np.sum(weights * N2 * (-ry / r2)) * S[j] / (2*pi)

            v2 = np.sum(weights * N2 * (rx / r2)) * S[j] / (2*pi)

            # Normal influence coefficients
            An[i, j] += u1*nx[i] + v1*ny[i]
            An[i, j+1] += u2*nx[i] + v2*ny[i]

            # Tangential influence coefficients
            At[i, j] += u1*tx[i] + v1*ty[i]
            At[i, j+1] += u2*tx[i] + v2*ty[i]

    # Freestream normal velocity
    rhs = -(U*nx + V*ny)

    # Construct complete system
    A = np.zeros((N+1, N+1))
    b = np.zeros(N+1)

    A[:N, :] = An
    b[:N] = rhs

    # Apply trailing-edge Kutta condition
    if kutta:

        A[N, 0] = 1
        A[N, N] = 1

    # For cylinder: impose zero circulation
    else:

        for j in range(N):

            A[N, j] += S[j] / 2
            A[N, j+1] += S[j] / 2

    # Solve for nodal vortex strengths
    gamma = np.linalg.solve(A, b)

    # Tangential velocity
    Vt = U*tx + V*ty + At @ gamma

    # Pressure coefficient
    Cp = 1 - (Vt/V_inf)**2

    # Pressure force integration
    Fx = -np.sum(Cp * nx * S)
    Fy = -np.sum(Cp * ny * S)

    # Lift coefficient
    chord = np.max(x) - np.min(x)

    Cl = (2 * (-Fx*np.sin(alpha) + Fy*np.cos(alpha))/ (V_inf**2 * chord))

    # Total circulation
    circulation = np.sum(S * (gamma[:-1] + gamma[1:]) / 2)

    return {
        "gamma": gamma,
        "Vt": Vt,
        "Cp": Cp,
        "Cl": Cl,
        "circulation": circulation,
        "xc": xc,
        "yc": yc,
        "S": S,
        "theta": theta
    }